# **Semana 14: Gestión de Errores y Arquitecturas Híbridas (NB1 - Conceptual)**

## **Propósito de la Sesión**
Integrar múltiples motores de bases de datos (SQL + NoSQL + Vectorial) en una arquitectura coherente y resiliente. Construiremos un sistema que combina SQLite como capa transaccional, un diccionario Python simulando Redis como capa de caché, y ChromaDB como capa de memoria semántica (embeddings). El flujo demostrará cómo mantener sincronizados estos motores ante operaciones de escritura.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Diseñar** una arquitectura híbrida que combine SQL, NoSQL (caché) y base vectorial.
2. **Implementar** una capa transaccional con SQLite para datos maestros.
3. **Simular** Redis con un diccionario Python para caché de alta velocidad.
4. **Utilizar** ChromaDB para almacenar y buscar embeddings de productos.
5. **Orquestar** un flujo donde al insertar un producto en SQLite se dispare automáticamente la actualización del embedding y la caché.
6. **Reflexionar** sobre los desafíos de consistencia y sincronización en sistemas políglotas.

## **Configuración Inicial**

Importamos las librerías necesarias y preparamos el entorno.

In [ ]:
# Instalación de librerías
!pip install chromadb sentence-transformers pandas --quiet

# Importaciones
import sqlite3
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import json
import time
import threading
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

print("Librerías importadas correctamente.")

---
## **Parte 1: Diseño de la Arquitectura Híbrida**

### **Componentes:**

| Capa | Tecnología | Propósito |
|------|------------|-----------|
| **Transaccional** | SQLite | Datos maestros (productos), ACID, consistencia fuerte |
| **Caché / Sesiones** | Diccionario Python (simula Redis) | Acceso rápido a productos frecuentes |
| **Memoria Semántica** | ChromaDB + Sentence Transformers | Búsqueda por similitud de productos |

### **Flujo de datos:**

1. **Escritura**: Se inserta/actualiza un producto en SQLite.
2. **Trigger simulado**: Un proceso detecta el cambio y:
   *   Actualiza la caché (Redis simulado).
   *   Genera/actualiza el embedding del producto y lo almacena en ChromaDB.
3. **Lectura**:
   *   **Consulta por ID**: Primero busca en caché, si no, en SQLite.
   *   **Búsqueda semántica**: Consulta directamente en ChromaDB.

Esta arquitectura refleja sistemas reales como e-commerce o recomendación de contenido.

---
## **Parte 2: Capa Transaccional - SQLite**

Crearemos la base de datos SQLite con una tabla de productos.

In [ ]:
# Conectar a SQLite (se crea el archivo si no existe)
conn_sqlite = sqlite3.connect('tienda.db', detect_types=sqlite3.PARSE_DECLTYPES)
cursor = conn_sqlite.cursor()

# Crear tabla de productos
cursor.execute('''
    CREATE TABLE IF NOT EXISTS productos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL,
        descripcion TEXT,
        precio REAL,
        categoria TEXT,
        fecha_creacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        fecha_actualizacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
''')

# Crear tabla para simular CDC (Change Data Capture) - registro de cambios
cursor.execute('''
    CREATE TABLE IF NOT EXISTS cambios_productos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        producto_id INTEGER,
        operacion TEXT, -- 'INSERT', 'UPDATE', 'DELETE'
        datos_json TEXT,
        fecha_cambio TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        procesado BOOLEAN DEFAULT 0
    )
''')

conn_sqlite.commit()
print("✅ Tablas creadas en SQLite.")

### **2.1. Función para insertar producto con trigger simulado**

En lugar de un trigger real de SQL, usaremos una función Python que inserta en `productos` y automáticamente registra el cambio en la tabla `cambios_productos`. Esto simula un trigger y nos da control sobre el flujo.

In [ ]:
def insertar_producto(nombre, descripcion, precio, categoria):
    """Inserta un producto y registra el cambio"""
    cursor = conn_sqlite.cursor()
    
    # Insertar producto
    cursor.execute('''
        INSERT INTO productos (nombre, descripcion, precio, categoria)
        VALUES (?, ?, ?, ?)
    ''', (nombre, descripcion, precio, categoria))
    
    producto_id = cursor.lastrowid
    
    # Registrar cambio
    datos = {
        'id': producto_id,
        'nombre': nombre,
        'descripcion': descripcion,
        'precio': precio,
        'categoria': categoria
    }
    cursor.execute('''
        INSERT INTO cambios_productos (producto_id, operacion, datos_json)
        VALUES (?, ?, ?)
    ''', (producto_id, 'INSERT', json.dumps(datos)))
    
    conn_sqlite.commit()
    return producto_id

# Insertar algunos productos de ejemplo
productos_ejemplo = [
    ("Laptop Gamer", "Laptop con RTX 3060, 16GB RAM", 1200, "Electrónica"),
    ("Mouse Inalámbrico", "Mouse ergonómico con batería de larga duración", 45, "Electrónica"),
    ("Teclado Mecánico", "Teclado con switches azules y retroiluminación RGB", 80, "Electrónica"),
    ("Monitor 4K", "Monitor UHD de 27 pulgadas para diseño", 350, "Electrónica"),
    ("Camiseta Deportiva", "Camiseta transpirable para running", 25, "Ropa")
]

for prod in productos_ejemplo:
    pid = insertar_producto(*prod)
    print(f"Producto insertado ID: {pid}")

# Verificar productos insertados
df_productos = pd.read_sql_query("SELECT * FROM productos", conn_sqlite)
print("\nProductos en SQLite:")
display(df_productos)

---
## **Parte 3: Capa de Caché - Simulación de Redis**

Simularemos Redis con un diccionario Python que incluye métodos básicos: `get`, `set`, `delete`, y `expire` (simulado).

In [ ]:
class RedisSimulado:
    def __init__(self):
        self._data = {}
        self._expires = {}
    
    def set(self, key, value, ttl=None):
        """Guarda un valor con TTL opcional (en segundos)"""
        self._data[key] = value
        if ttl:
            self._expires[key] = time.time() + ttl
        return True
    
    def get(self, key):
        """Obtiene un valor si existe y no ha expirado"""
        if key in self._expires and time.time() > self._expires[key]:
            # Ha expirado
            del self._data[key]
            del self._expires[key]
            return None
        return self._data.get(key)
    
    def delete(self, key):
        """Elimina una clave"""
        if key in self._data:
            del self._data[key]
        if key in self._expires:
            del self._expires[key]
        return True
    
    def exists(self, key):
        """Verifica si una clave existe y no ha expirado"""
        return self.get(key) is not None
    
    def keys(self, pattern=None):
        """Retorna todas las claves (simplificado)"""
        return list(self._data.keys())
    
    def flush(self):
        """Limpia toda la caché"""
        self._data.clear()
        self._expires.clear()

# Instancia global de Redis simulado
redis_cache = RedisSimulado()

print("✅ Redis simulado inicializado.")

---
## **Parte 4: Capa Vectorial - ChromaDB**

Configuramos ChromaDB para almacenar embeddings de productos y permitir búsqueda semántica.

In [ ]:
# Inicializar ChromaDB en memoria
chroma_client = chromadb.Client(Settings(
    chroma_db_impl="duckdb+parquet",
    persist_directory="./chroma_persist"  # Para persistencia
))

# Crear colección para productos
try:
    chroma_client.delete_collection("productos")
except:
    pass

collection = chroma_client.create_collection(
    name="productos",
    metadata={"hnsw:space": "cosine"}
)

print("✅ Colección 'productos' creada en ChromaDB.")

# Cargar modelo de embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✅ Modelo de embeddings cargado. Dimensión: {model.get_sentence_embedding_dimension()}")

---
## **Parte 5: Orquestación - Worker de Sincronización**

Crearemos un worker que periódicamente revisa la tabla `cambios_productos` y sincroniza los cambios con Redis (caché) y ChromaDB (embeddings).

In [ ]:
def sincronizar_cambios():
    """Procesa los cambios pendientes y actualiza Redis y ChromaDB"""
    cursor = conn_sqlite.cursor()
    
    # Obtener cambios no procesados
    cursor.execute('''
        SELECT id, producto_id, operacion, datos_json
        FROM cambios_productos
        WHERE procesado = 0
        ORDER BY id ASC
    ''')
    
    cambios = cursor.fetchall()
    
    for cambio_id, producto_id, operacion, datos_json in cambios:
        datos = json.loads(datos_json)
        
        print(f"\n⚙️ Procesando cambio ID {cambio_id}: {operacion} producto {producto_id}")
        
        # 1. Actualizar Redis (caché)
        if operacion in ('INSERT', 'UPDATE'):
            # Guardar en Redis con TTL de 1 hora (3600s)
            redis_cache.set(f"producto:{producto_id}", json.dumps(datos), ttl=3600)
            print(f"  ✅ Redis: producto {producto_id} cacheado")
        elif operacion == 'DELETE':
            redis_cache.delete(f"producto:{producto_id}")
            print(f"  ✅ Redis: producto {producto_id} eliminado de caché")
        
        # 2. Actualizar ChromaDB (embeddings)
        if operacion in ('INSERT', 'UPDATE'):
            # Generar embedding combinando nombre y descripción
            texto = f"{datos['nombre']} {datos.get('descripcion', '')} {datos.get('categoria', '')}"
            embedding = model.encode(texto).tolist()
            
            # Preparar metadatos
            metadata = {
                'nombre': datos['nombre'],
                'precio': datos['precio'],
                'categoria': datos.get('categoria', ''),
                'operacion': operacion,
                'timestamp': datetime.now().isoformat()
            }
            
            # Insertar o actualizar en Chroma (upsert)
            collection.upsert(
                ids=[str(producto_id)],
                embeddings=[embedding],
                metadatas=[metadata],
                documents=[texto]
            )
            print(f"  ✅ ChromaDB: embedding actualizado para producto {producto_id}")
            
        elif operacion == 'DELETE':
            # Eliminar de Chroma si existe
            try:
                collection.delete(ids=[str(producto_id)])
                print(f"  ✅ ChromaDB: producto {producto_id} eliminado")
            except:
                print(f"  ⚠️ ChromaDB: producto {producto_id} no encontrado para eliminar")
        
        # Marcar cambio como procesado
        cursor.execute('UPDATE cambios_productos SET procesado = 1 WHERE id = ?', (cambio_id,))
        conn_sqlite.commit()
    
    return len(cambios)

# Procesar cambios iniciales (productos recién insertados)
procesados = sincronizar_cambios()
print(f"\n🔄 Sincronización inicial completada. {procesados} cambios procesados.")

---
## **Parte 6: Verificación de la Sincronización**

Comprobaremos que los datos están disponibles en las tres capas.

### **6.1. Verificar Redis (caché)**

In [ ]:
print("--- CONTENIDO DE REDIS (CACHÉ) ---")
for key in redis_cache.keys():
    if key.startswith('producto:'):
        valor = redis_cache.get(key)
        print(f"{key}: {json.loads(valor)['nombre']}")

### **6.2. Verificar ChromaDB (embeddings)**

In [ ]:
print("\n--- CONTENIDO DE CHROMADB ---")
resultados = collection.get(limit=10)
for i, (doc_id, metadata) in enumerate(zip(resultados['ids'], resultados['metadatas'])):
    print(f"{i+1}. ID: {doc_id} - {metadata['nombre']} (cat: {metadata['categoria']})")

---
## **Parte 7: Búsqueda Semántica en ChromaDB**

Probemos la capacidad de búsqueda por similitud.

In [ ]:
def buscar_semantico(consulta, k=3):
    """Busca productos similares a la consulta"""
    print(f"\n🔍 Consulta: '{consulta}'")
    embedding = model.encode([consulta]).tolist()[0]
    
    results = collection.query(
        query_embeddings=[embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    print("\nResultados:")
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        sim = 1 - distance  # para cosine distance
        print(f"  {i+1}. [Sim: {sim:.4f}] {metadata['nombre']} (${metadata['precio']})")
        print(f"     {doc[:100]}...")

buscar_semantico("computadora portátil para juegos")
buscar_semantico("ropa deportiva cómoda")
buscar_semantico("teclado con luces")

---
## **Parte 8: Flujo Completo - Insertar y Verificar Sincronización**

Ahora insertaremos un nuevo producto y verificaremos que automáticamente se sincroniza con Redis y ChromaDB.

In [ ]:
print("--- NUEVO PRODUCTO ---")
nuevo_id = insertar_producto(
    "Tablet Profesional",
    "Tablet con lápiz táctil, 256GB, ideal para diseñadores",
    650,
    "Electrónica"
)
print(f"Producto insertado con ID: {nuevo_id}")

# Ejecutar sincronización
print("\n🔄 Ejecutando sincronización...")
sincronizar_cambios()

# Verificar en Redis
print("\n--- VERIFICACIÓN EN REDIS ---")
cached = redis_cache.get(f"producto:{nuevo_id}")
if cached:
    prod = json.loads(cached)
    print(f"Producto en caché: {prod['nombre']} - ${prod['precio']}")

# Verificar en Chroma con búsqueda semántica
buscar_semantico("tableta para dibujar", k=5)

---
## **Parte 9: Simulación de Fallo y Recuperación**

Simularemos un fallo en la caché (Redis simulado) y veremos cómo el sistema puede recuperarse desde SQLite.

In [ ]:
def obtener_producto(producto_id):
    """Obtiene producto con estrategia caché primero"""
    # Intentar desde caché
    cached = redis_cache.get(f"producto:{producto_id}")
    if cached:
        print(f"✅ CACHE HIT - Producto {producto_id}")
        return json.loads(cached)
    
    print(f"❌ CACHE MISS - Producto {producto_id}, consultando SQLite...")
    # Consultar SQLite
    cursor = conn_sqlite.cursor()
    cursor.execute('SELECT * FROM productos WHERE id = ?', (producto_id,))
    row = cursor.fetchone()
    
    if row:
        producto = {
            'id': row[0],
            'nombre': row[1],
            'descripcion': row[2],
            'precio': row[3],
            'categoria': row[4]
        }
        # Opcional: actualizar caché
        redis_cache.set(f"producto:{producto_id}", json.dumps(producto), ttl=3600)
        return producto
    return None

# Vaciar caché para simular fallo
redis_cache.flush()
print("🧹 Caché vaciada (simulando fallo).\n")

# Probar obtención
prod = obtener_producto(1)
print(f"Producto obtenido: {prod['nombre']}")

# Segunda llamada (debería venir de caché)
prod2 = obtener_producto(1)
print(f"Segunda llamada: {prod2['nombre']}")

---
## **Ejercicios Prácticos para el Estudiante**

Ahora extiende y mejora la arquitectura.

### **Ejercicio 1: Actualizar producto**

Implementa una función `actualizar_producto` que:
1. Actualice un producto en SQLite.
2. Registre el cambio en `cambios_productos` (operación 'UPDATE').
3. Ejecute la sincronización y verifique que la caché y ChromaDB se actualizan.

*Pista: Similar a `insertar_producto` pero con UPDATE.*

In [ ]:
# --- INICIO DE TU CÓDIGO ---

def actualizar_producto(producto_id, nombre=None, descripcion=None, precio=None, categoria=None):
    # 1. Obtener datos actuales
    # ...
    # 2. Actualizar en SQLite
    # ...
    # 3. Registrar cambio
    # ...
    # 4. Sincronizar
    # ...
    pass

# Prueba: actualizar el producto con ID 3
# actualizar_producto(3, precio=75)

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Búsqueda híbrida con filtro de precio**

ChromaDB permite filtrar por metadatos antes de la búsqueda vectorial. Modifica la función `buscar_semantico` para aceptar un filtro de precio mínimo y máximo.

Ejemplo: buscar productos similares a "computadora" con precio entre 500 y 1000.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

def buscar_semantico_filtrado(consulta, precio_min=None, precio_max=None, k=3):
    # Construir filtro
    filtro = {}
    if precio_min is not None:
        filtro["precio"] = {"$gte": precio_min}
    if precio_max is not None:
        filtro["precio"] = {"$lte": precio_max}
    # ...
    pass

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: Worker en segundo plano**

Modifica el worker de sincronización para que se ejecute en un hilo en segundo plano, revisando cambios cada 5 segundos. Añade logs para ver cuándo procesa cambios.

*Pista: Usa `threading` y `time.sleep`.*

In [ ]:
# --- INICIO DE TU CÓDIGO ---

def worker_loop():
    while True:
        try:
            cambios = sincronizar_cambios()
            if cambios:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Procesados {cambios} cambios.")
            time.sleep(5)
        except Exception as e:
            print(f"Error en worker: {e}")
            time.sleep(5)

# Iniciar worker (daemon para que termine al cerrar)
# worker = threading.Thread(target=worker_loop, daemon=True)
# worker.start()

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 4: Reflexión final del curso**

En una celda Markdown, responde:
1. ¿Qué ventajas ofrece una arquitectura híbrida (SQL + NoSQL + Vectorial) frente a usar un solo motor?
2. ¿Qué desafíos de consistencia y sincronización introduces al usar múltiples motores?
3. ¿Cómo aplicarías esta arquitectura en un caso real (ej. sistema de recomendación de películas, e-commerce, chatbot)?
4. ¿Qué estrategias de recuperación ante desastres implementarías en producción?

---
## **Conclusiones del Curso**

A lo largo de 14 semanas, hemos recorrido el camino desde los fundamentos de las bases de datos hasta las arquitecturas más avanzadas:

*   **Semanas 1-4**: Modelado relacional, normalización, SQL.
*   **Semanas 5-7**: Índices, consultas optimizadas, limpieza de datos.
*   **Semanas 8-9**: Captura de datos, pipelines ETL/ELT, batch vs streaming.
*   **Semanas 10-12**: Big Data (Spark, Databricks), NoSQL (MongoDB, Redis, Neo4j), bases vectoriales (ChromaDB, Pinecone) y RAG.
*   **Semanas 13-14**: Despliegue con Docker, alta disponibilidad, y arquitecturas híbridas.

**Hoy hemos integrado todo:**
1. **SQLite** como capa transaccional ACID.
2. **Redis simulado** como capa de caché de baja latencia.
3. **ChromaDB** como capa de memoria semántica para búsqueda por similitud.
4. **Sincronización** automática mediante CDC simulado.

Este es el tipo de arquitectura que encontrarás en aplicaciones modernas de IA: sistemas políglotas que aprovechan lo mejor de cada motor.

In [ ]:
# Cerrar conexiones
conn_sqlite.close()
print("\nConexiones cerradas. ¡Fin del curso!")